# Étape 5 — Transformer de type GPT : prédire la suite d'un texte

```
Input  : "Artificial Intelligence can ..."
              │
              ▼
   ┌───────────────────────────────┐
   │ Generative Pre-trained        │
   │ Transformer (GPT-2)           │   décodeur uniquement, attention causale
   └───────────────────────────────┘
              │
              ▼
Output : "... help people and organizations ..."
```

GPT est **autorégressif** : il prédit le token suivant à partir des tokens précédents uniquement,
l'ajoute au texte, puis recommence.

> Le premier lancement télécharge le modèle `gpt2` (~500 Mo) depuis Hugging Face.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

## 1. Charger le tokenizer et le modèle préentraîné

In [2]:
nom_modele = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForCausalLM.from_pretrained(nom_modele)

# GPT-2 n'a pas de token de padding : on réutilise le token de fin de texte.
tokenizer.pad_token = tokenizer.eos_token
modele.config.pad_token_id = tokenizer.pad_token_id
modele.eval()

print("Paramètres :", f"{sum(p.numel() for p in modele.parameters()) / 1e6:.0f} M")
print("Blocs Transformer :", modele.config.n_layer, "| têtes :", modele.config.n_head,
      "| dimension :", modele.config.n_embd)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\yassi\Downloads\tp-architecture-transformer\tp-architecture-transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yassi\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Paramètres : 124 M
Blocs Transformer : 12 | têtes : 12 | dimension : 768


## 2. Transformer le texte en tokens

In [3]:
texte = "Artificial intelligence can"
entrees = tokenizer(texte, return_tensors="pt")

print("Dimensions de l'entrée :", entrees["input_ids"].shape)   # (1, nombre de tokens)
print("Tokens :", tokenizer.convert_ids_to_tokens(entrees["input_ids"][0]))

Dimensions de l'entrée : torch.Size([1, 4])
Tokens : ['Art', 'ificial', 'Ġintelligence', 'Ġcan']


## 3. Ce que GPT calcule : une distribution sur le prochain token

`logits` a la forme `(1, L, taille_vocabulaire)`. Seule la **dernière position** sert à prédire le token suivant.

In [4]:
with torch.no_grad():
    logits = modele(**entrees).logits

print("Logits :", tuple(logits.shape))
probas = torch.softmax(logits[0, -1], dim=-1)
top = torch.topk(probas, k=5)

print(f"\nTokens les plus probables après « {texte} » :")
for p, i in zip(top.values, top.indices):
    print(f"  {p.item():6.2%}  -> {tokenizer.decode([i.item()])!r}")

Logits : (1, 4, 50257)

Tokens les plus probables après « Artificial intelligence can » :
  15.29%  -> ' be'
   7.19%  -> ' also'
   7.05%  -> ' help'
   2.80%  -> "'t"
   2.52%  -> ' do'


## 4. Générer la suite, un token après l'autre

In [5]:
def generer(texte, max_new_tokens=40, **kwargs):
    entrees = tokenizer(texte, return_tensors="pt")
    with torch.no_grad():
        sortie = modele.generate(**entrees, max_new_tokens=max_new_tokens,
                                 pad_token_id=tokenizer.pad_token_id, **kwargs)
    return tokenizer.decode(sortie[0], skip_special_tokens=True)


print("== Glouton (toujours le token le plus probable) ==")
print(generer("Artificial intelligence can help people", do_sample=False))

print("\n== Échantillonnage (temperature=0.8, top_p=0.9) ==")
torch.manual_seed(0)
print(generer("Artificial intelligence can help people", do_sample=True, temperature=0.8, top_p=0.9))

== Glouton (toujours le token le plus probable) ==


Artificial intelligence can help people to make better decisions, but it's not going to solve all of the problems that we face today.

The problem is that we're not going to solve all of the problems that we face

== Échantillonnage (temperature=0.8, top_p=0.9) ==


Artificial intelligence can help people understand what their behavior is like and how they respond to it. This helps the machine understand what is happening around them and to be able to make more effective decisions.

However, when you look


## 5. Effet de la température

In [6]:
for t in [0.3, 0.8, 1.5]:
    torch.manual_seed(1)
    print(f"--- temperature = {t} ---")
    print(generer("Artificial intelligence can help people", max_new_tokens=30,
                  do_sample=True, temperature=t, top_p=0.95))
    print()

--- temperature = 0.3 ---


Artificial intelligence can help people understand what they are doing and how they are doing it.

The problem is that it is not easy to understand what is happening in the real

--- temperature = 0.8 ---


Artificial intelligence can help people understand what makes them happy and what makes them miserable, and it can make them less susceptible to social exclusion, and it can help people understand how others

--- temperature = 1.5 ---


Artificial intelligence can help people understand what makes them happy and happy for different times; it does so in order to prevent problems in different ways and so far the techniques can produce outcomes



## Conclusion

- GPT-2 est un **Transformer décodeur** : chaque position ne voit que les positions précédentes (masque causal).
- La génération est **autorégressive** : prédire → ajouter → recommencer.
- Température basse → texte prudent et répétitif ; température haute → texte plus varié mais moins cohérent.
- GPT-2 est entraîné principalement sur de l'anglais : les prompts en français donnent des résultats plus faibles.

➡️ Étape suivante : `06_bert.ipynb`